## Processed Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/shuffled_10_data.csv")

print(df.shape)
print(df.columns.tolist())
df.head(3)

(53430, 24)
['AC', 'PMID', 'Title', 'Abstract', 'Terms', 'Title_clean', 'Abstract_clean', 'Text_combined', 'Term_list', 'batch_number', 'autoactivation', 'autocatalysis', 'autocatalytic', 'autofeedback', 'autoinducer', 'autoinduction', 'autoinhibition', 'autoinhibitory', 'autokinase', 'autolysis', 'autophosphorylation', 'autoregulation', 'autoregulatory', 'autoubiquitination']


,AC,PMID,Title,Abstract,Terms,Title_clean,Abstract_clean,Text_combined,Term_list,batch_number,...,autoinducer,autoinduction,autoinhibition,autoinhibitory,autokinase,autolysis,autophosphorylation,autoregulation,autoregulatory,autoubiquitination
0,O54965,19292867,The PA-TM-RING protein RING finger protein 13 ...,PA-TM-RING proteins have an N-terminal proteas...,autoubiquitination,The PA-TM-RING protein RING finger protein 13 ...,PA-TM-RING proteins have an N-terminal proteas...,The PA-TM-RING protein RING finger protein 13 ...,['autoubiquitination'],1,...,0,0,0,0,0,0,0,0,0,1
1,P29323,36805027,E3 ligase autoinhibition by C-degron mimicry m...,E3 ligase recruitment of proteins containing t...,autoinhibition,E3 ligase autoinhibition by C-degron mimicry m...,E3 ligase recruitment of proteins containing t...,E3 ligase autoinhibition by C-degron mimicry m...,['autoinhibition'],1,...,0,0,1,0,0,0,0,0,0,0
2,A7Z6W9,17704766,Comparative analysis of the complete genome se...,Bacillus amyloliquefaciens FZB42 is a Gram-pos...,NaN,Comparative analysis of the complete genome se...,Bacillus amyloliquefaciens FZB42 is a Gram-pos...,Comparative analysis of the complete genome se...,[],1,...,0,0,0,0,0,0,0,0,0,0


## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

## Load and Split Data

In [4]:
df = pd.read_csv("../data/processed/shuffled_10_data.csv")
df = df[df["batch_number"].isin([1, 2])].copy()
text_col = "Text_combined"
label_cols = df.columns[df.columns.get_loc("Term_list") + 1:]

df1 = df[df["batch_number"] == 1]
train_df, val_df = train_test_split(df1, test_size=0.2, random_state=42)

## Define Dataset

In [5]:
class BertMultiLabelDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.texts = dataframe[text_col].tolist()
        self.labels = dataframe[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

## Define Model

In [6]:
class BertMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        logits = self.classifier(x)
        return logits

## Prepare Data

In [9]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_ds = BertMultiLabelDataset(train_df, tokenizer)
val_ds = BertMultiLabelDataset(val_df, tokenizer)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16)

## Initialize Model

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

## Training Function

In [8]:
def train_epoch(dataloader):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(dataloader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.append((probs > 0.5).astype(int))
            targets.append(labels)

    preds = np.vstack(preds)
    targets = np.vstack(targets)
    micro_f1 = f1_score(targets, preds, average="micro", zero_division=0)
    return micro_f1

## Training Loop

In [9]:
for epoch in range(3):
    loss = train_epoch(train_dl)
    f1 = evaluate(val_dl)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val Micro F1: {f1:.4f}")

Training:   0%|          | 0/268 [00:00<?, ?it/s]c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Training: 100%|██████████| 268/268 [04:21<00:00,  1.02it/s]


Epoch 1 | Loss: 0.1770 | Val Micro F1: 0.8778


Training: 100%|██████████| 268/268 [16:39<00:00,  3.73s/it]


Epoch 2 | Loss: 0.0734 | Val Micro F1: 0.9028


Training: 100%|██████████| 268/268 [21:17<00:00,  4.77s/it]


Epoch 3 | Loss: 0.0501 | Val Micro F1: 0.9178


## Naive BERT

In [9]:
# Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel, BertConfig
from tqdm import tqdm

In [10]:
# Set Device and Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [11]:
# Label Columns (Explicit)
label_cols = [
    'autoactivation', 'autocatalysis', 'autocatalytic', 'autofeedback', 'autoinducer',
    'autoinduction', 'autoinhibition', 'autoinhibitory', 'autokinase', 'autolysis',
    'autophosphorylation', 'autoregulation', 'autoregulatory', 'autoubiquitination'
]

In [12]:
# Dataset Class
class BertMultiLabelDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.texts = dataframe["Text_combined"].tolist()
        self.labels = dataframe[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

In [13]:
# Model Definition: Untrained BERT
class BertMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        config = BertConfig()
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        return self.classifier(x)

In [14]:
# Utility Functions
def compute_pos_weights(df_subset):
    pos_weights = []
    for col in label_cols:
        pos = (df_subset[col] == 1).sum()
        neg = (df_subset[col] == 0).sum()
        w = neg / pos if pos > 0 else 1.0
        pos_weights.append(w)
    return torch.tensor(pos_weights, dtype=torch.float32).to(device)

def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy().astype(int)
            labels = np.clip(labels, 0, 1)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.append((probs > 0.5).astype(int))
            targets.append(labels)
    preds = np.vstack(preds)
    targets = np.vstack(targets)
    return f1_score(targets, preds, average="micro", zero_division=0)

In [15]:
# Load Data
df = pd.read_csv("../data/processed/shuffled_10_data.csv")
results = {}

In [16]:
# Train BATCH 1
print("\n==== BATCH 1 ====")
df1 = df[df["batch_number"] == 1].copy()
train_df1, val_df1 = train_test_split(df1, test_size=0.2, random_state=42)

pos_weights_1 = compute_pos_weights(train_df1)
train_ds1 = BertMultiLabelDataset(train_df1, tokenizer)
val_ds1   = BertMultiLabelDataset(val_df1, tokenizer)
train_dl1 = DataLoader(train_ds1, batch_size=16, shuffle=True)
val_dl1   = DataLoader(val_ds1, batch_size=16)

model1 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
criterion1 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_1)
optimizer1 = optim.Adam(model1.parameters(), lr=2e-5)

for epoch in range(3):
    loss1 = train_one_epoch(model1, train_dl1, optimizer1, criterion1)
    f1_1 = evaluate_model(model1, val_dl1)
    print(f"[Batch 1] Epoch {epoch+1} | Loss: {loss1:.4f} | Val Micro F1: {f1_1:.4f}")
results["batch1_f1"] = f1_1

# === Train BATCH 2 ===
print("\n==== BATCH 2 ====")
df2 = df[df["batch_number"] == 2].copy()
train_df2, val_df2 = train_test_split(df2, test_size=0.2, random_state=42)

pos_weights_2 = compute_pos_weights(train_df2)
train_ds2 = BertMultiLabelDataset(train_df2, tokenizer)
val_ds2   = BertMultiLabelDataset(val_df2, tokenizer)
train_dl2 = DataLoader(train_ds2, batch_size=16, shuffle=True)
val_dl2   = DataLoader(val_ds2, batch_size=16)

model2 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
criterion2 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_2)
optimizer2 = optim.Adam(model2.parameters(), lr=2e-5)

for epoch in range(3):
    loss2 = train_one_epoch(model2, train_dl2, optimizer2, criterion2)
    f1_2 = evaluate_model(model2, val_dl2)
    print(f"[Batch 2] Epoch {epoch+1} | Loss: {loss2:.4f} | Val Micro F1: {f1_2:.4f}")
results["batch2_f1"] = f1_2


==== BATCH 1 ====


Training:   0%|          | 0/268 [00:00<?, ?it/s]c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Training: 100%|██████████| 268/268 [04:22<00:00,  1.02it/s]


[Batch 1] Epoch 1 | Loss: 1.4269 | Val Micro F1: 0.0279


Training: 100%|██████████| 268/268 [04:22<00:00,  1.02it/s]


[Batch 1] Epoch 2 | Loss: 1.4415 | Val Micro F1: 0.0717


...

[Batch 2] Epoch 1 | Loss: 1.4590 | Val Micro F1: 0.0301


Training: 100%|██████████| 268/268 [22:21<00:00,  5.01s/it]


[Batch 2] Epoch 2 | Loss: 1.4260 | Val Micro F1: 0.0287


Training: 100%|██████████| 268/268 [29:03<00:00,  6.51s/it]


[Batch 2] Epoch 3 | Loss: 1.4031 | Val Micro F1: 0.0222


In [ ]:
# Summary
print("\n=== Final Micro F1 Scores ===")
print(f"Batch 1: {results['batch1_f1']:.4f}")
print(f"Batch 2: {results['batch2_f1']:.4f}")

## bert-base-uncased

In [ ]:
# Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [ ]:
# Device & Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:

# Explicit Label Columns
label_cols = [
    'autoactivation', 'autocatalysis', 'autocatalytic', 'autofeedback', 'autoinducer',
    'autoinduction', 'autoinhibition', 'autoinhibitory', 'autokinase', 'autolysis',
    'autophosphorylation', 'autoregulation', 'autoregulatory', 'autoubiquitination'
]

In [ ]:
# Dataset Class
class BertMultiLabelDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.texts = dataframe["Text_combined"].tolist()
        self.labels = dataframe[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

In [ ]:
# BERT Class
class BertMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        return self.classifier(x)

In [ ]:
# Calculating pos_weight
def compute_pos_weights(df_subset):
    pos_weights = []
    for col in label_cols:
        pos = (df_subset[col] == 1).sum()
        neg = (df_subset[col] == 0).sum()
        w = neg / pos if pos > 0 else 1.0
        pos_weights.append(w)
    return torch.tensor(pos_weights, dtype=torch.float32).to(device)

In [ ]:
# Training and Evaluation
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy().astype(int)
            labels = np.clip(labels, 0, 1)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.append((probs > 0.5).astype(int))
            targets.append(labels)
    preds = np.vstack(preds)
    targets = np.vstack(targets)
    return f1_score(targets, preds, average="micro", zero_division=0)

In [ ]:
# Read Data
df = pd.read_csv("../data/processed/shuffled_10_data.csv")
results = {}

In [ ]:
# Batch 1
print("\nTraining on Batch 1")
df1 = df[df["batch_number"] == 1].copy()
train_df1, val_df1 = train_test_split(df1, test_size=0.2, random_state=42)
pos_weights_1 = compute_pos_weights(train_df1)

train_ds1 = BertMultiLabelDataset(train_df1, tokenizer)
val_ds1   = BertMultiLabelDataset(val_df1, tokenizer)
train_dl1 = DataLoader(train_ds1, batch_size=16, shuffle=True)
val_dl1   = DataLoader(val_ds1, batch_size=16)

model1 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
criterion1 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_1)
optimizer1 = optim.Adam(model1.parameters(), lr=2e-5)

for epoch in range(3):
    loss1 = train_one_epoch(model1, train_dl1, optimizer1, criterion1)
    f1_1 = evaluate_model(model1, val_dl1)
    print(f"[Batch 1] Epoch {epoch+1} | Loss: {loss1:.4f} | Val Micro F1: {f1_1:.4f}")
results["batch1_f1"] = f1_1

# Batch 2
print("\nTraining on Batch 2")
df2 = df[df["batch_number"] == 2].copy()
train_df2, val_df2 = train_test_split(df2, test_size=0.2, random_state=42)
pos_weights_2 = compute_pos_weights(train_df2)

train_ds2 = BertMultiLabelDataset(train_df2, tokenizer)
val_ds2   = BertMultiLabelDataset(val_df2, tokenizer)
train_dl2 = DataLoader(train_ds2, batch_size=16, shuffle=True)
val_dl2   = DataLoader(val_ds2, batch_size=16)

model2 = BertMultiLabelClassifier(num_labels=len(label_cols)).to(device)
criterion2 = nn.BCEWithLogitsLoss(pos_weight=pos_weights_2)
optimizer2 = optim.Adam(model2.parameters(), lr=2e-5)

for epoch in range(3):
    loss2 = train_one_epoch(model2, train_dl2, optimizer2, criterion2)
    f1_2 = evaluate_model(model2, val_dl2)
    print(f"[Batch 2] Epoch {epoch+1} | Loss: {loss2:.4f} | Val Micro F1: {f1_2:.4f}")
results["batch2_f1"] = f1_2

In [ ]:
# Summary
print("\n=== Final Micro F1 Scores ===")
print(f"Batch 1: {results['batch1_f1']:.4f}")
print(f"Batch 2: {results['batch2_f1']:.4f}")